# Strata Colab Quickstart — Run Qwen3.8 / Qwen3.6 on Google Colab GPUs

Run the **[Strata](https://github.com/Niko1221/Strata)** hybrid GPU/RAM inference engine directly inside a **Google Colab GPU** notebook (`G4`, `H100`, `A100`, `L4`, or `T4`) using the verified v0.2.19+ CLI flags and OpenAI-compatible `/v1` API.

- **Auto GPU Model Selection (`MODEL_PRESET = "Auto (match GPU)"`)**:
  - **G4 (98 GB) / H100 (80 GB) / A100 High-RAM (80 GB)**: `Qwen3.8-Flash-Next` (`IQ3_S`)
  - **A100 Standard (40 GB)**: `Qwen3.8-Flash-Next` (`IQ2_XS`)
  - **L4 (23 GB) / T4 (15 GB)**: `Qwen3.6-35B-A3B` (`UD-Q4_K_XL`)
- **Shared Google Drive Cache (`/content/drive/MyDrive/Strata-data`)**: Compatible with **[Strata Colab Desktop](https://github.com/yarik1390/strata-colab-desktop)** (`start.bat` / `./up.sh`) so downloaded models can be reused between this notebook and the full desktop setup.
- **Storage & Compute Units**: Verify at least **60–120 GB** of free disk space depending on the selected model (`~23 GB` for `UD-Q4_K_XL`, `~68–84 GB` for `IQ2_XS` / `IQ3_S`). Use Colab strictly for **personal interactive research and benchmarking** and delete your runtime when finished.

References: [Strata](https://github.com/Niko1221/Strata) | [Strata Colab Desktop](https://github.com/yarik1390/strata-colab-desktop) | [Google Colab FAQ](https://research.google.com/colaboratory/faq.html)

## 1. Check GPU, RAM & Disk and Clone Strata

Select your preferred model preset (or keep **`Auto (match GPU)`**) and context length, then run this cell to inspect your Colab hardware, choose an available loopback port (`8089`, `8081`, `8082`, or `8080`), install `uv`, and clone Strata.

In [ ]:
import os, re, shutil, socket, subprocess, sys
from pathlib import Path

MODEL_PRESET = "Auto (match GPU)"  # @param ["Auto (match GPU)", "Qwen3.8-Flash-Next (IQ3_S)", "Qwen3.8-Flash-Next (IQ2_XS)", "Qwen3.8-Flash-Next (Q2_0)", "Qwen3.8-Coder (IQ1_M)", "Qwen3.6-35B-A3B (UD-Q4_K_XL)"]
CONTEXT = "32768"  # @param ["32768", "65536", "131072", "262144"]

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["free", "-h"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)

gpu_line = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
    text=True,
).strip().splitlines()[0]
gpu_name, vram_str, driver = [x.strip() for x in gpu_line.split(",")[:3]]
vram_mib = int(float(vram_str))

mem_total_kb = 0
with open("/proc/meminfo", encoding="utf-8") as f:
    for line in f:
        if line.startswith("MemTotal:"):
            mem_total_kb = int(line.split()[1])
            break
ram_gib = mem_total_kb / (1024 * 1024)

PRESET_MAP = {
    "Qwen3.8-Flash-Next (IQ3_S)": ("", "IQ3_S"),
    "Qwen3.8-Flash-Next (IQ2_XS)": ("", "IQ2_XS"),
    "Qwen3.8-Flash-Next (Q2_0)": ("", "Q2_0"),
    "Qwen3.8-Coder (IQ1_M)": ("coder", "IQ1_M"),
    "Qwen3.6-35B-A3B (UD-Q4_K_XL)": ("qwen36", "UD-Q4_K_XL"),
}

if MODEL_PRESET == "Auto (match GPU)":
    if vram_mib >= 70000 and ram_gib >= 120:
        MODEL_FAMILY, MODEL = "", "IQ3_S"
    elif vram_mib >= 38000:
        MODEL_FAMILY, MODEL = "", "IQ2_XS"
    else:
        MODEL_FAMILY, MODEL = "qwen36", "UD-Q4_K_XL"
else:
    MODEL_FAMILY, MODEL = PRESET_MAP[MODEL_PRESET]

print(f"Detected GPU: {gpu_name} ({vram_mib} MiB VRAM, driver {driver}), System RAM: {ram_gib:.1f} GiB")
print(f"Selected preset: {MODEL_PRESET} -> family={MODEL_FAMILY or 'default'}, model={MODEL}, context={CONTEXT}")


def port_is_free(p: int) -> bool:
    with socket.socket() as sock:
        try:
            sock.bind(("127.0.0.1", p))
            return True
        except OSError:
            return False


PORT = next((p for p in (8089, 8081, 8082, 8080) if port_is_free(p)), None)
assert PORT is not None, "No free loopback port found among 8089, 8081, 8082, 8080"
os.environ["STRATA_PORT"] = str(PORT)
print("Selected Strata port:", PORT)

if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

root = Path("/content/Strata")
if not (root / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Niko1221/Strata.git", str(root)], check=True)

print("Repository ready. Proceed to the next step.")

## 2. [Optional] Restore Cached Model from Google Drive

If you previously cached model weights in `/content/drive/MyDrive/Strata-data` (either from this notebook or from `strata-colab-desktop`), run this cell **before** `setup.sh` to restore them onto Colab's local NVMe SSD at datacenter speed.

In [ ]:
import subprocess
from pathlib import Path
from google.colab import drive

DRIVE_CACHE_DIR = "/content/drive/MyDrive/Strata-data"  # @param {type:"string"}

drive.mount("/content/drive")
source = Path(DRIVE_CACHE_DIR)
if source.is_dir():
    dest = Path("/content/Strata-data")
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "rsync", "-a", "--info=progress2",
            "--exclude=strata-config/",
            "--exclude=conversation-spill/",
            "--exclude=*.part",
            "--exclude=*.tmp",
            str(source) + "/",
            str(dest) + "/",
        ],
        check=True,
    )
    print("Drive cache restore complete:", dest)
else:
    print("No existing Drive cache found at", source, "- setup will download weights fresh.")

## 3. Install Strata and Prepare the Quantized Model

Uses `STRATA_UV=1` and verified `setup.sh` flags with `--no-browser --yes --no-start`. Output streams live below while Strata sets up its Python environment, downloads or verifies GGUF weights, and prepares runtime files.

In [ ]:
import os, subprocess
from pathlib import Path

os.environ["STRATA_UV"] = "1"
os.environ["PATH"] = (
    "/usr/local/bin:" + os.path.expanduser("~/.local/bin")
    + ":" + os.environ.get("PATH", "")
)
env = os.environ.copy()

cmd = [
    "bash", "setup.sh",
    "--model", MODEL,
    "--context", str(CONTEXT),
    "--port", str(PORT),
    "--data-dir", "/content/Strata-data",
    "--no-browser", "--yes", "--no-start",
]
if MODEL_FAMILY:
    cmd += ["--family", MODEL_FAMILY]

print("Running:", " ".join(cmd), flush=True)
proc = subprocess.Popen(
    cmd,
    cwd="/content/Strata",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace",
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"setup.sh exited with status {rc}")

candidates = list(Path("/content/Strata/terminal").glob("run-*.sh")) + list(Path("/content/Strata").glob("run-*.sh"))
exact = [p for p in candidates if p.name.lower() == f"run-{MODEL.lower()}.sh"]
partial = [p for p in candidates if MODEL.lower() in p.name.lower()]
pool = exact or partial or candidates
pool.sort(key=lambda p: p.stat().st_mtime, reverse=True)

assert pool, "Setup completed but no run-*.sh launch script was found under /content/Strata"
RUN_SCRIPT = pool[0]
print("Setup complete. Launch script:", RUN_SCRIPT)

## 4. Start the Strata Server and Wait for the Model to Load

Launches the generated `run-*.sh` script in the background and polls `http://127.0.0.1:{PORT}/health` (and `/v1/models`) up to 10 minutes until the model weights are loaded into VRAM/RAM.

In [ ]:
import subprocess, time, requests
from pathlib import Path

BASE = f"http://127.0.0.1:{PORT}"
s = requests.Session()
s.trust_env = False


def get_status():
    health = None
    models = []
    try:
        r = s.get(BASE + "/health", timeout=3)
        if r.ok:
            j = r.json()
            if j.get("service") == "strata" or "loaded" in j or "status" in j:
                health = j
    except (requests.RequestException, ValueError):
        pass
    try:
        mr = s.get(BASE + "/v1/models", timeout=3)
        if mr.ok:
            mj = mr.json()
            models = [m["id"] for m in mj.get("data", []) if isinstance(m, dict) and m.get("id")]
    except (requests.RequestException, ValueError):
        pass
    return health, models


health, models = get_status()
if not ((health and health.get("loaded")) or models):
    assert RUN_SCRIPT.is_file(), f"Launch script missing: {RUN_SCRIPT}"
    log_path = Path(f"/content/strata-{PORT}.log")
    with log_path.open("a", encoding="utf-8") as log:
        proc = subprocess.Popen(
            ["bash", str(RUN_SCRIPT)],
            cwd=str(RUN_SCRIPT.parent),
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    print(f"Server started (PID={proc.pid}). Loading model weights...")
    for i in range(120):  # up to 10 minutes
        health, models = get_status()
        if (health and health.get("loaded")) or models:
            break
        if proc.poll() is not None:
            print(log_path.read_text(errors="replace")[-5000:])
            raise RuntimeError(f"Strata server exited unexpectedly (exit {proc.returncode})")
        if i % 6 == 0:
            print(f"{i * 5}s elapsed. Waiting for model to finish loading...")
        time.sleep(5)
    else:
        print(log_path.read_text(errors="replace")[-5000:])
        raise TimeoutError("Strata did not become ready within 10 minutes. Check the log above.")

ACTIVE_MODEL_ID = (health or {}).get("model") or (models[0] if models else f"qwen3.8-flash-next-{MODEL.lower()}")
print("READY:", health or {"models": models})
print("OpenAI-compatible API:", BASE + "/v1")
print("Active model ID:", ACTIVE_MODEL_ID)

## 5. Stream Text Generation & Display Speed Metrics

Streams a 512-token response from `/v1/chat/completions` and reports prompt tokens, output tokens, Prefill `tok/s`, Decode `tok/s`, time to first token, and speculative decoding acceptance rate (when active).

In [ ]:
import json, time

payload = {
    "model": ACTIVE_MODEL_ID,
    "messages": [
        {
            "role": "user",
            "content": "Explain with concrete examples how AI is transforming R&D at chemical manufacturers.",
        }
    ],
    "max_tokens": 512,
    "reasoning_effort": "none",
    "temperature": 0,
    "stream": True,
}

start = time.perf_counter()
first_text = None
usage, timings = {}, {}
print("=== Text Generation (Streaming) ===\n")
with s.post(BASE + "/v1/chat/completions", json=payload, stream=True, timeout=(10, 900)) as r:
    if not r.ok:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:800]}")
    for line in r.iter_lines(chunk_size=1):
        if not line or not line.startswith(b"data:"):
            continue
        raw = line[5:].strip()
        if raw == b"[DONE]":
            break
        event = json.loads(raw)
        if event.get("error"):
            raise RuntimeError(str(event["error"]))
        for choice in event.get("choices", []):
            delta = choice.get("delta") or {}
            piece = delta.get("content") or delta.get("reasoning_content") or ""
            if piece:
                if first_text is None:
                    first_text = time.perf_counter()
                print(piece, end="", flush=True)
        usage = event.get("usage") or usage
        timings = event.get("timings") or timings

elapsed = time.perf_counter() - start
print("\n\n=== Benchmark ===")
print("Prompt tokens:", usage.get("prompt_tokens"))
print("Output tokens:", usage.get("completion_tokens"))
print("Prefill:", timings.get("prompt_per_second"), "tok/s")
print("Decode:", timings.get("predicted_per_second"), "tok/s")
print("Total time:", round(elapsed, 2), "s")
if first_text:
    print("Time to first token:", round(first_text - start, 2), "s")
if timings.get("draft_n"):
    accept = timings.get("draft_n_accepted", 0)
    print("Speculative decoding acceptance rate:", round(100 * accept / timings["draft_n"], 1), "%")

## 6. [Optional] Compare Long-Context Prefill Performance

Runs single-shot summarization prompts across **Short (100 chars)**, **Medium (20,000 chars)**, and **Long (50,000 chars)** inputs (`max_tokens=64`) to compare prefill throughput across prompt lengths.

In [ ]:
import time, uuid

paragraph = (
    "Artificial intelligence is transforming scientific research. "
    "Researchers use machine learning to analyze experimental data, "
    "discover new materials, optimize chemical processes, and predict "
    "physical properties. Reliable evaluation requires reproducible "
    "experiments, careful validation, and quantitative comparisons. "
)

for label, chars in [("Short", 100), ("Medium", 20000), ("Long", 50000)]:
    doc = (paragraph * (chars // len(paragraph) + 1))[:chars]
    text = f"Test ID: {uuid.uuid4().hex}\nSummarize the following text concisely.\n{doc}"
    data = {
        "model": ACTIVE_MODEL_ID,
        "messages": [{"role": "user", "content": text}],
        "max_tokens": 64,
        "reasoning_effort": "none",
        "temperature": 0,
        "stream": False,
    }
    t0 = time.perf_counter()
    r = s.post(BASE + "/v1/chat/completions", json=data, timeout=900)
    r.raise_for_status()
    out = r.json()
    tm, usage = out.get("timings", {}), out.get("usage", {})
    print(
        f"{label}: prompt {usage.get('prompt_tokens')} tokens | "
        f"Prefill {tm.get('prompt_per_second')} tok/s | "
        f"Decode {tm.get('predicted_per_second')} tok/s | "
        f"Cache reuse {tm.get('cache_n')} | "
        f"Total time {time.perf_counter() - t0:.2f} s"
    )

## 7. [Optional] Back Up the Model Cache to Google Drive

Run after model downloads and preparation finish so future sessions (including `strata-colab-desktop`) can restore weights directly from Google Drive instead of re-downloading.

In [ ]:
import subprocess
from pathlib import Path
from google.colab import drive

DRIVE_CACHE_DIR = "/content/drive/MyDrive/Strata-data"  # @param {type:"string"}

drive.mount("/content/drive")
backup = Path(DRIVE_CACHE_DIR)
backup.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        "rsync", "-a", "--info=progress2",
        "--exclude=conversation-spill/",
        "--exclude=*.part",
        "--exclude=*.tmp",
        "/content/Strata-data/",
        str(backup) + "/",
    ],
    check=True,
)
print("Model cache backup complete:", backup)

---

**When finished:** Select **Runtime → Disconnect and delete runtime** to stop Compute Unit consumption.

**Credits:** Powered by **[Strata](https://github.com/Niko1221/Strata)** by [@Niko1221](https://github.com/Niko1221) (MIT License), maintained in **[Strata Colab Desktop](https://github.com/yarik1390/strata-colab-desktop)**.